In [1]:
import pandas as pd
import numpy as np

# Display options to keep output clean and readable
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: '%.2f' % x)

# Path to dataset on D drive structure
data_path = '../data/raw/online_retail_II.csv'

# Read CSV file
df = pd.read_csv(data_path, low_memory=False)

print("Dataset loaded successfully.")

Dataset loaded successfully.


In [2]:
# Display shape
print(f"Dataset Shape: {df.shape[0]:,} rows and {df.shape[1]} columns")

# View first 5 rows
print("\n--- First 5 Rows ---")
display(df.head())

# View columns and data types
print("\n--- Column Info ---")
df.info()

Dataset Shape: 1,067,371 rows and 8 columns

--- First 5 Rows ---


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.00,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.00,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.00,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.00,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.00,United Kingdom



--- Column Info ---
<class 'pandas.DataFrame'>
RangeIndex: 1067371 entries, 0 to 1067370
Data columns (total 8 columns):
 #   Column       Non-Null Count    Dtype  
---  ------       --------------    -----  
 0   Invoice      1067371 non-null  str    
 1   StockCode    1067371 non-null  str    
 2   Description  1062989 non-null  str    
 3   Quantity     1067371 non-null  int64  
 4   InvoiceDate  1067371 non-null  str    
 5   Price        1067371 non-null  float64
 6   Customer ID  824364 non-null   float64
 7   Country      1067371 non-null  str    
dtypes: float64(2), int64(1), str(5)
memory usage: 65.1 MB


In [3]:
# Missing values per column
missing_df = pd.DataFrame({
    'Missing_Count': df.isnull().sum(),
    'Missing_Pct': (df.isnull().sum() / len(df)) * 100
})

print("--- Missing Values ---")
print(missing_df)

# Check exact duplicate rows
duplicate_rows = df.duplicated().sum()
print(f"\nExact Duplicate Rows: {duplicate_rows:,} ({(duplicate_rows/len(df))*100:.2f}%)")

--- Missing Values ---
             Missing_Count  Missing_Pct
Invoice                  0         0.00
StockCode                0         0.00
Description           4382         0.41
Quantity                 0         0.00
InvoiceDate              0         0.00
Price                    0         0.00
Customer ID         243007        22.77
Country                  0         0.00

Exact Duplicate Rows: 34,335 (3.22%)


In [4]:
# Convert InvoiceDate to datetime to compute exact range
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

print("--- Key Entity Counts ---")
print(f"Unique Invoices: {df['Invoice'].nunique():,}")
print(f"Unique Stock Codes (Products): {df['StockCode'].nunique():,}")
print(f"Unique Customers: {df['Customer ID'].nunique():,}")
print(f"Unique Countries: {df['Country'].nunique()}")

print("\n--- Date Range ---")
print(f"Start Date: {df['InvoiceDate'].min()}")
print(f"End Date:   {df['InvoiceDate'].max()}")

--- Key Entity Counts ---
Unique Invoices: 53,628
Unique Stock Codes (Products): 5,305
Unique Customers: 5,942
Unique Countries: 43

--- Date Range ---
Start Date: 2009-12-01 07:45:00
End Date:   2011-12-09 12:50:00


In [5]:
# Identify cancellations (Invoice starting with 'C')
cancelled_invoices = df['Invoice'].astype(str).str.startswith('C').sum()

# Identify negative quantities
negative_qty = (df['Quantity'] < 0).sum()

# Identify zero or negative prices
zero_or_neg_price = (df['Price'] <= 0).sum()

print("--- Data Anomalies ---")
print(f"Cancelled Invoice Records (starts with 'C'): {cancelled_invoices:,}")
print(f"Negative Quantity Records:                 {negative_qty:,}")
print(f"Zero or Negative Price Records:            {zero_or_neg_price:,}")

--- Data Anomalies ---
Cancelled Invoice Records (starts with 'C'): 19,494
Negative Quantity Records:                 22,950
Zero or Negative Price Records:            6,207


In [6]:
# Statistical overview of quantity and unit price
print("--- Numerical Distribution ---")
display(df[['Quantity', 'Price']].describe(percentiles=[0.01, 0.05, 0.50, 0.95, 0.99]))

--- Numerical Distribution ---


,Quantity,Price
count,1067371.00,1067371.00
mean,9.94,4.65
std,172.71,123.55
min,-80995.00,-53594.36
1%,-3.00,0.21
5%,1.00,0.42
50%,3.00,2.10
95%,30.00,9.95
99%,100.00,18.00
max,80995.00,38970.00


In [7]:
# Print data dictionary table based on calculated attributes
data_dict = pd.DataFrame({
    'Column Name': df.columns,
    'Data Type': [str(dtype) for dtype in df.dtypes],
    'Non-Null Count': df.notnull().sum().values,
    'Sample Value': [df[col].dropna().iloc[0] for col in df.columns],
    'Description': [
        'Unique 6-digit transaction number (prefixed with C if cancelled)',
        'Unique product identification code',
        'Name/description of product',
        'Quantity of product units per order line item',
        'Date and time of order generation',
        'Unit price of product in GBP (£)',
        'Unique customer identifier',
        'Country of purchase'
    ]
})

print("--- Data Dictionary ---")
display(data_dict)

--- Data Dictionary ---


,Column Name,Data Type,Non-Null Count,Sample Value,Description
0,Invoice,str,1067371,489434,Unique 6-digit transaction number (prefixed wi...
1,StockCode,str,1067371,85048,Unique product identification code
2,Description,str,1062989,15CM CHRISTMAS GLASS BALL 20 LIGHTS,Name/description of product
3,Quantity,int64,1067371,12,Quantity of product units per order line item
4,InvoiceDate,datetime64[us],1067371,2009-12-01 07:45:00,Date and time of order generation
5,Price,float64,1067371,6.95,Unit price of product in GBP (£)
6,Customer ID,float64,824364,13085.00,Unique customer identifier
7,Country,str,1067371,United Kingdom,Country of purchase
